# Exercice intégré, partie 4 : Spark (couches silver et gold)

Suivi d'expéditions de la Control Tower GlobalTech. Date d'observation : **23 novembre 2026 à 00:00**.

Chemin des données dans le data lake HDFS :

| Zone | Chemin | Contenu |
|---|---|---|
| bronze | `/datalake/bronze/expeditions/ingestion_date=2026-11-23/` | `expeditions_v1.jsonl`, tel que reçu |
| silver | `/datalake/silver/expeditions` | événements valides, dédoublonnés, sans donnée personnelle (Parquet) |
| rejects | `/datalake/rejects/expeditions` | lignes rejetées, avec leur motif (Parquet) |
| gold | `/datalake/gold/expeditions`, `/datalake/gold/otd_transporteur` | une ligne par expédition ; OTD par transporteur (Parquet) |

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Sur Colab, fs.defaultFS suit la checklist (souvent hdfs://localhost:9000) ; ici le NameNode écoute sur le port 8020.
spark = (SparkSession.builder.master("local[*]").appName("expeditions-silver-gold")
         .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:8020")
         .config("spark.sql.session.timeZone", "UTC")   # horodatages sans fuseau : on les garde tels quels
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BRONZE = "/datalake/bronze/expeditions/ingestion_date=2026-11-23/expeditions_v1.jsonl"
print("Spark", spark.version)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=44323 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=
Picked up _JAVA_OPTIONS: -XX:+UnlockDiagnosticVMOptions -XX:DisableIntrinsic=_base64_encodeBlock,_base64_decodeBlock
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.ne

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 23:09:03 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/25 23:09:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 23:09:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.1


## 4.1 Lecture avec un schéma explicite

On impose le schéma au lieu de laisser Spark le deviner : les types sont garantis, la lecture est plus rapide (pas de passe d'inférence), et une colonne inattendue ne change pas silencieusement la structure.
`ts_evenement` est lu **en texte** : certaines lignes ont un horodatage `null` ou vide, qu'on veut repérer et rejeter avec un motif, pas perdre sans le savoir.

In [2]:
schema = StructType([
    StructField("id_expedition", StringType()),
    StructField("statut", StringType()),
    StructField("ts_evenement", StringType()),
    StructField("transporteur", StringType()),
    StructField("fournisseur", StringType()),
    StructField("pays_origine", StringType()),
    StructField("pays_dest", StringType()),
    StructField("delai_promis_j", IntegerType()),
    StructField("poids_kg", DoubleType()),
    StructField("contact_email", StringType()),
])
brut = spark.read.schema(schema).json(BRONZE)
nb_brut = brut.count()
print("Lignes brutes :", nb_brut)
brut.printSchema()
brut.select("id_expedition", "statut", "ts_evenement", "transporteur", "delai_promis_j").show(5)

Lignes brutes : 3944
root
 |-- id_expedition: string (nullable = true)
 |-- statut: string (nullable = true)
 |-- ts_evenement: string (nullable = true)
 |-- transporteur: string (nullable = true)
 |-- fournisseur: string (nullable = true)
 |-- pays_origine: string (nullable = true)
 |-- pays_dest: string (nullable = true)
 |-- delai_promis_j: integer (nullable = true)
 |-- poids_kg: double (nullable = true)
 |-- contact_email: string (nullable = true)



+-------------+------+--------------------+-------------------+--------------+
|id_expedition|statut|        ts_evenement|       transporteur|delai_promis_j|
+-------------+------+--------------------+-------------------+--------------+
|  EXP-H-00287|  CREE|2026-09-01 04:22:...|TransEurope Express|             9|
|  EXP-H-01168|  CREE|2026-09-01 04:51:...|           MedCargo|             6|
|  EXP-H-00917|  CREE|2026-09-01 06:10:...|TransEurope Express|             2|
|  EXP-H-00648|  CREE|2026-09-01 06:44:...|         RapidRoute|             6|
|  EXP-H-00146|  CREE|2026-09-01 06:46:...|         RapidRoute|             2|
+-------------+------+--------------------+-------------------+--------------+
only showing top 5 rows


## 4.2 Couche silver

Règles, dans cet ordre :
1. **rejet** des lignes au statut invalide (hors `CREE`, `EN_TRANSIT`, `EN_DOUANE`, `LIVRE`), puis de celles sans horodatage, avec un **motif** ;
2. **suppression des doublons exacts** parmi les lignes valides ;
3. **suppression de `contact_email`** (donnée personnelle, inutile pour le KPI).

Les rejets sont écrits à part : on garde la trace de ce qui a été écarté et pourquoi.

In [3]:
STATUTS_VALIDES = ["CREE", "EN_TRANSIT", "EN_DOUANE", "LIVRE"]

avec_motif = (brut
    # try_to_timestamp renvoie NULL sur un texte vide ou mal formé. to_timestamp lèverait une erreur
    # en Spark 4 (mode ANSI activé par défaut) ; en Spark 3, to_timestamp renvoyait NULL.
    .withColumn("ts", F.try_to_timestamp("ts_evenement", F.lit("yyyy-MM-dd HH:mm:ss.SSS")))
    .withColumn("motif_rejet",
        F.when(F.col("statut").isNull() | ~F.col("statut").isin(STATUTS_VALIDES), F.lit("statut_invalide"))
         .when(F.col("ts").isNull(), F.lit("horodatage_manquant"))))

rejets = avec_motif.filter(F.col("motif_rejet").isNotNull()).drop("ts", "contact_email")
valides = avec_motif.filter(F.col("motif_rejet").isNull()).drop("motif_rejet")

silver = (valides.dropDuplicates()                       # doublons exacts
          .drop("ts_evenement", "contact_email")          # texte d'origine et donnée personnelle retirés
          .withColumnRenamed("ts", "ts_evenement"))       # horodatage typé (timestamp)

nb_rejets, nb_valides, nb_silver = rejets.count(), valides.count(), silver.count()
nb_doublons = nb_valides - nb_silver
print(f"Rejets : {nb_rejets} | lignes valides : {nb_valides} | silver : {nb_silver} | doublons supprimés : {nb_doublons}")
rejets.groupBy("motif_rejet", "statut").count().orderBy("motif_rejet", F.desc("count")).show()

Rejets : 77 | lignes valides : 3867 | silver : 3790 | doublons supprimés : 77


+-------------------+----------+-----+
|        motif_rejet|    statut|count|
+-------------------+----------+-----+
|horodatage_manquant|EN_TRANSIT|   10|
|horodatage_manquant|     LIVRE|   10|
|horodatage_manquant|      CREE|    8|
|horodatage_manquant| EN_DOUANE|    3|
|    statut_invalide|   INCONNU|   13|
|    statut_invalide|     livre|   10|
|    statut_invalide|          |    9|
|    statut_invalide|    ANNULE|    8|
|    statut_invalide|     LIVRÉ|    6|
+-------------------+----------+-----+



In [4]:
print(f"Contrôle : brut {nb_brut} = silver {nb_silver} + rejets {nb_rejets} + doublons {nb_doublons} = {nb_silver + nb_rejets + nb_doublons}")
assert nb_brut == nb_silver + nb_rejets + nb_doublons, "le bilan ne tombe pas juste"

silver.write.mode("overwrite").parquet("/datalake/silver/expeditions")
rejets.write.mode("overwrite").parquet("/datalake/rejects/expeditions")
silver.printSchema()
silver.show(3)

Contrôle : brut 3944 = silver 3790 + rejets 77 + doublons 77 = 3944


root
 |-- id_expedition: string (nullable = true)
 |-- statut: string (nullable = true)
 |-- transporteur: string (nullable = true)
 |-- fournisseur: string (nullable = true)
 |-- pays_origine: string (nullable = true)
 |-- pays_dest: string (nullable = true)
 |-- delai_promis_j: integer (nullable = true)
 |-- poids_kg: double (nullable = true)
 |-- ts_evenement: timestamp (nullable = true)



+-------------+------+------------+-----------------+------------+---------+--------------+--------+--------------------+
|id_expedition|statut|transporteur|      fournisseur|pays_origine|pays_dest|delai_promis_j|poids_kg|        ts_evenement|
+-------------+------+------------+-----------------+------------+---------+--------------+--------+--------------------+
|  EXP-H-00573|  CREE|    MedCargo|     Iberia Plast|          ES|       FR|             4|  1460.0|2026-09-10 01:45:...|
|  EXP-H-00809|  CREE|  RapidRoute|Polska Komponenty|          PL|       FR|             4|  1408.7|2026-10-09 21:46:...|
|  EXP-H-00182|  CREE|    MedCargo|     Iberia Plast|          ES|       FR|             6|  2089.2|2026-10-28 07:32:...|
+-------------+------+------------+-----------------+------------+---------+--------------+--------+--------------------+
only showing top 3 rows


## 4.3 Couche gold : une ligne par expédition

- `date_creation` : horodatage de l'événement `CREE` ; `date_livraison` : celui de `LIVRE` (vide si non livrée) ;
- `duree_reelle_j` = (livraison − création) en jours ;
- `livree_a_l_heure` = durée réelle ≤ délai promis.

**OTD (On-Time Delivery)** = livrées à l'heure ÷ livrées.

**Hypothèse** : une expédition livrée dont l'événement `CREE` a été rejeté (statut invalide ou horodatage manquant) n'a pas de date de création fiable. On ne peut pas calculer sa durée : elle est **exclue du calcul de l'OTD** et comptée à part.

In [5]:
silver = spark.read.parquet("/datalake/silver/expeditions")

gold = (silver.groupBy("id_expedition")
    .agg(F.min(F.when(F.col("statut") == "CREE", F.col("ts_evenement"))).alias("date_creation"),
         F.max(F.when(F.col("statut") == "LIVRE", F.col("ts_evenement"))).alias("date_livraison"),
         F.first("delai_promis_j").alias("delai_promis_j"),
         F.first("transporteur").alias("transporteur"),
         F.first("fournisseur").alias("fournisseur"),
         F.first("pays_origine").alias("pays_origine"))
    .withColumn("duree_reelle_j", (F.unix_timestamp("date_livraison") - F.unix_timestamp("date_creation")) / 86400)
    .withColumn("livree", F.col("date_livraison").isNotNull())
    .withColumn("livree_a_l_heure",
                F.when(F.col("livree") & F.col("date_creation").isNotNull(),
                       F.col("duree_reelle_j") <= F.col("delai_promis_j"))))

nb_exp = gold.count()
nb_livrees = gold.filter("livree").count()
nb_sans_creation = gold.filter("livree AND date_creation IS NULL").count()
print(f"Expéditions : {nb_exp} | livrées : {nb_livrees} | non livrées : {nb_exp - nb_livrees}")
print(f"Livrées sans date de création fiable (exclues de l'OTD) : {nb_sans_creation}")
gold.select("id_expedition", "transporteur", "date_creation", "date_livraison", "delai_promis_j",
            F.round("duree_reelle_j", 2).alias("duree_reelle_j"), "livree_a_l_heure").show(5)

Expéditions : 1199 | livrées : 984 | non livrées : 215
Livrées sans date de création fiable (exclues de l'OTD) : 19


+-------------+------------------+--------------------+--------------------+--------------+--------------+----------------+
|id_expedition|      transporteur|       date_creation|      date_livraison|delai_promis_j|duree_reelle_j|livree_a_l_heure|
+-------------+------------------+--------------------+--------------------+--------------+--------------+----------------+
|  EXP-H-00001|          MedCargo|2026-11-18 00:13:...|                NULL|             6|          NULL|            NULL|
|  EXP-H-00002|AsiaLink Logistics|2026-11-03 09:23:...|2026-11-13 07:29:...|            18|          9.92|            true|
|  EXP-H-00003|AsiaLink Logistics|2026-10-03 13:43:...|2026-10-31 13:23:...|            28|         27.99|            true|
|  EXP-H-00004|          MedCargo|2026-10-15 02:54:...|2026-10-22 11:33:...|            12|          7.36|            true|
|  EXP-H-00005|          MedCargo|2026-10-05 14:55:...|2026-10-14 13:42:...|            11|          8.95|            true|
+-------

In [6]:
base_otd = gold.filter("livree AND date_creation IS NOT NULL")

def otd(df, *cles):
    agg = df.groupBy(*cles) if cles else df.groupBy()
    return (agg.agg(F.sum(F.col("livree_a_l_heure").cast("int")).alias("livrees_a_l_heure"),
                    F.count("*").alias("livrees"))
               .withColumn("otd_pct", F.round(100 * F.col("livrees_a_l_heure") / F.col("livrees"), 1)))

otd_global = otd(base_otd)
otd_transporteur = otd(base_otd, "transporteur").orderBy(F.desc("otd_pct"))
print("OTD global :")
otd_global.show()
print("OTD par transporteur :")
otd_transporteur.show()

gold.write.mode("overwrite").parquet("/datalake/gold/expeditions")
otd_transporteur.write.mode("overwrite").parquet("/datalake/gold/otd_transporteur")

OTD global :


+-----------------+-------+-------+
|livrees_a_l_heure|livrees|otd_pct|
+-----------------+-------+-------+
|              773|    965|   80.1|
+-----------------+-------+-------+

OTD par transporteur :


+-------------------+-----------------+-------+-------+
|       transporteur|livrees_a_l_heure|livrees|otd_pct|
+-------------------+-----------------+-------+-------+
|           NordFret|              137|    140|   97.9|
|         RapidRoute|              141|    165|   85.5|
|TransEurope Express|              183|    222|   82.4|
|           MedCargo|              180|    252|   71.4|
| AsiaLink Logistics|              132|    186|   71.0|
+-------------------+-----------------+-------+-------+



## 4.4 Analyse complémentaire

Avant de conclure sur les transporteurs, on vérifie qu'on compare des choses comparables : ils n'ont pas tous les mêmes routes. Les trajets **hors Union européenne** passent par la douane et sont plus longs.

In [7]:
PAYS_UE = ["FR", "DE", "IT", "ES", "PL", "BE", "NL"]
par_zone = base_otd.withColumn("zone", F.when(F.col("pays_origine").isin(PAYS_UE), "UE").otherwise("hors UE"))
otd(par_zone, "transporteur", "zone").orderBy("transporteur", "zone").show()

print("Retard moyen des livraisons en retard (jours au-delà du délai promis) :")
(base_otd.filter(~F.col("livree_a_l_heure"))
    .groupBy("transporteur")
    .agg(F.count("*").alias("livraisons_en_retard"),
         F.round(F.avg(F.col("duree_reelle_j") - F.col("delai_promis_j")), 1).alias("retard_moyen_j"))
    .orderBy(F.desc("retard_moyen_j")).show())

+-------------------+-------+-----------------+-------+-------+
|       transporteur|   zone|livrees_a_l_heure|livrees|otd_pct|
+-------------------+-------+-----------------+-------+-------+
| AsiaLink Logistics|hors UE|              132|    186|   71.0|
|           MedCargo|     UE|               75|    110|   68.2|
|           MedCargo|hors UE|              105|    142|   73.9|
|           NordFret|     UE|               94|     96|   97.9|
|           NordFret|hors UE|               43|     44|   97.7|
|         RapidRoute|     UE|              141|    165|   85.5|
|TransEurope Express|     UE|              141|    172|   82.0|
|TransEurope Express|hors UE|               42|     50|   84.0|
+-------------------+-------+-----------------+-------+-------+

Retard moyen des livraisons en retard (jours au-delà du délai promis) :


+-------------------+--------------------+--------------+
|       transporteur|livraisons_en_retard|retard_moyen_j|
+-------------------+--------------------+--------------+
| AsiaLink Logistics|                  54|           9.0|
|           NordFret|                   3|           3.8|
|           MedCargo|                  72|           2.3|
|         RapidRoute|                  24|           1.6|
|TransEurope Express|                  39|           1.3|
+-------------------+--------------------+--------------+



In [8]:
# Relecture de la couche gold depuis HDFS, pour vérifier ce qui a été écrit
spark.read.parquet("/datalake/gold/otd_transporteur").orderBy(F.desc("otd_pct")).show()
print("Lignes dans /datalake/gold/expeditions :", spark.read.parquet("/datalake/gold/expeditions").count())

+-------------------+-----------------+-------+-------+
|       transporteur|livrees_a_l_heure|livrees|otd_pct|
+-------------------+-----------------+-------+-------+
|           NordFret|              137|    140|   97.9|
|         RapidRoute|              141|    165|   85.5|
|TransEurope Express|              183|    222|   82.4|
|           MedCargo|              180|    252|   71.4|
| AsiaLink Logistics|              132|    186|   71.0|
+-------------------+-----------------+-------+-------+



Lignes dans /datalake/gold/expeditions : 1199


### Interprétation (4.4)

| Transporteur | OTD | Constat |
|---|---|---|
| NordFret | 97,9 % | Excellent, en Europe (97,9 %) comme sur les États-Unis (97,7 %) |
| RapidRoute | 85,5 % | Correct, uniquement en Europe |
| TransEurope Express | 82,4 % | Correct, en Europe comme hors Europe |
| MedCargo | 71,4 % | Faible **en Europe (68,2 %) comme hors Europe (73,9 %)** : le problème vient du transporteur, pas de la route |
| AsiaLink Logistics | 71,0 % | Le plus faible sur les trajets hors Europe, et des retards longs (9 jours en moyenne quand il est en retard) |

**Décisions proposées** :
1. **MedCargo** : plan d'amélioration avec pénalités de retard, et transfert progressif d'une partie de ses volumes européens (Italie, Espagne) vers NordFret ou RapidRoute, qui desservent déjà ces pays avec un meilleur OTD.
2. **AsiaLink Logistics** : avant de sanctionner, vérifier si les délais promis sur la Chine et l'Inde (25 à 35 jours) sont réalistes et si les retards viennent de la douane (durée entre `EN_DOUANE` et `LIVRE`). Mettre en concurrence au moins un second transporteur sur ces routes.
3. **NordFret** : transporteur de référence ; lui confier davantage de volume là où il opère.

Limites : 19 expéditions livrées sans date de création fiable sont exclues du calcul (0,9 point d'OTD au plus) ; l'OTD ne compte que les expéditions **livrées**, il ignore les 215 non livrées, dont certaines sont déjà en retard.